# Weekly Hierarchical Demand Forecasting

This notebook forecasts demand at the more stable **Product Sub-Group** level and allocates each forecast back to SKUs using a leakage-free historical demand mix.

Workflow:

1. Aggregate daily demand into complete weeks.
2. Hold out the final four weeks for validation.
3. Compare previous-week, four-week moving-average, and eight-week moving-average forecasts.
4. Select the best model using pooled Sub-Group WAPE.
5. Allocate Sub-Group forecasts to SKUs using 80% recent-eight-week share plus 20% long-run share.
6. Validate reconciled SKU forecasts and generate the next two weeks.

**Bias convention:** positive forecast bias means over-forecasting; negative bias means under-forecasting.

In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = next(
    (
        path
        for path in (Path.cwd().resolve(), *Path.cwd().resolve().parents)
        if (path / "config.py").is_file()
    ),
    None,
)
if PROJECT_ROOT is None:
    raise FileNotFoundError(
        "Could not find config.py. Start Jupyter from the repository or its Notebooks folder."
    )
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import numpy as np
import pandas as pd

from config import (
    DAILY_FLOW_FILE,
    FORECAST_TOPDOWN_FILE,
    HOLDOUT_WEEKS,
    HORIZON_WEEKS,
    PROCESSED_DIR,
    RECENT_SHARE_WEIGHT,
    RECENT_WEEKS,
    WEEK_FREQUENCY,
    ensure_output_directories,
)

INPUT_FILE = DAILY_FLOW_FILE
OUTPUT_DIR = PROCESSED_DIR
FORECAST_HORIZON_WEEKS = HORIZON_WEEKS
TEST_WEEKS = HOLDOUT_WEEKS
RECENT_SHARE_WEEKS = RECENT_WEEKS

ensure_output_directories()
print("Input:", INPUT_FILE)
print("Output directory:", OUTPUT_DIR)


## 1. Load data and create complete weekly SKU demand

In [ ]:
df = pd.read_csv(INPUT_FILE)
df["Date"] = pd.to_datetime(df["Date"])

required_columns = {
    "Date", "product_id", "Group", "Sub-Group", "sales_units"
}
missing_columns = required_columns.difference(df.columns)
if missing_columns:
    raise ValueError(f"Missing required columns: {sorted(missing_columns)}")

if df.duplicated(["Date", "product_id"]).any():
    raise ValueError("Duplicate Date + product_id keys found in daily_flow.csv")

if (df["sales_units"] < 0).any():
    raise ValueError("Negative sales_units found")

# Weeks end on Wednesday so the final observed week (2023-08-03 to 2023-08-09) is complete.
df["week_end"] = (
    df["Date"].dt.to_period(WEEK_FREQUENCY).dt.end_time.dt.normalize()
)

week_day_counts = df.groupby("week_end")["Date"].nunique()
complete_weeks = week_day_counts[week_day_counts == 7].index
weekly_source = df[df["week_end"].isin(complete_weeks)].copy()

weekly_sku = (
    weekly_source.groupby(
        ["week_end", "Group", "Sub-Group", "product_id"],
        as_index=False,
    )["sales_units"].sum()
)

weekly_subgroup = (
    weekly_sku.groupby(
        ["week_end", "Group", "Sub-Group"],
        as_index=False,
    )["sales_units"].sum()
    .rename(columns={"sales_units": "subgroup_sales_units"})
)

print("Complete weeks:", weekly_sku["week_end"].nunique())
print("Products:", weekly_sku["product_id"].nunique())
print("Sub-Groups:", weekly_sku["Sub-Group"].nunique())
print("Range:", weekly_sku["week_end"].min(), "to", weekly_sku["week_end"].max())
weekly_sku.head()

## 2. Forecast functions and leakage-free SKU allocation shares

In [ ]:
def forecast_weekly(history, horizon, model):
    history = np.asarray(history, dtype=float)

    if len(history) == 0:
        return np.zeros(horizon)
    if model == "previous_week":
        return np.repeat(history[-1], horizon)
    if model == "moving_average_4":
        return np.repeat(history[-min(4, len(history)):].mean(), horizon)
    if model == "moving_average_8":
        return np.repeat(history[-min(8, len(history)):].mean(), horizon)
    raise ValueError(f"Unknown model: {model}")


def build_sku_shares(historical_sku, recent_weeks=8, recent_weight=0.80):
    # Build shares using only the supplied historical period.
    # Fall back to long-run share when recent demand is zero, and to equal
    # shares when the full Sub-Group history is also zero.
    product_dim = historical_sku[
        ["Group", "Sub-Group", "product_id"]
    ].drop_duplicates()

    full_totals = (
        historical_sku.groupby(
            ["Group", "Sub-Group", "product_id"], as_index=False
        )["sales_units"].sum()
        .rename(columns={"sales_units": "full_sku_sales"})
    )

    available_weeks = sorted(historical_sku["week_end"].unique())
    selected_recent_weeks = available_weeks[-recent_weeks:]
    recent_totals = (
        historical_sku[
            historical_sku["week_end"].isin(selected_recent_weeks)
        ]
        .groupby(["Group", "Sub-Group", "product_id"], as_index=False)[
            "sales_units"
        ].sum()
        .rename(columns={"sales_units": "recent_sku_sales"})
    )

    shares = (
        product_dim
        .merge(full_totals, on=["Group", "Sub-Group", "product_id"], how="left")
        .merge(recent_totals, on=["Group", "Sub-Group", "product_id"], how="left")
    )
    shares[["full_sku_sales", "recent_sku_sales"]] = shares[
        ["full_sku_sales", "recent_sku_sales"]
    ].fillna(0)

    group_keys = ["Group", "Sub-Group"]
    shares["full_subgroup_sales"] = shares.groupby(group_keys)[
        "full_sku_sales"
    ].transform("sum")
    shares["recent_subgroup_sales"] = shares.groupby(group_keys)[
        "recent_sku_sales"
    ].transform("sum")
    shares["sku_count"] = shares.groupby(group_keys)["product_id"].transform("count")

    shares["full_share"] = np.where(
        shares["full_subgroup_sales"] > 0,
        shares["full_sku_sales"] / shares["full_subgroup_sales"],
        1 / shares["sku_count"],
    )
    shares["recent_share"] = np.where(
        shares["recent_subgroup_sales"] > 0,
        shares["recent_sku_sales"] / shares["recent_subgroup_sales"],
        shares["full_share"],
    )
    shares["allocation_share"] = (
        recent_weight * shares["recent_share"]
        + (1 - recent_weight) * shares["full_share"]
    )
    shares["allocation_share"] = shares["allocation_share"] / shares.groupby(
        group_keys
    )["allocation_share"].transform("sum")

    return shares[
        ["Group", "Sub-Group", "product_id", "allocation_share"]
    ]


MODELS = ["previous_week", "moving_average_4", "moving_average_8"]

## 3. Four-week holdout: select the best Sub-Group model

In [ ]:
all_weeks = sorted(weekly_sku["week_end"].unique())
if len(all_weeks) <= TEST_WEEKS + 8:
    raise ValueError("Not enough complete weeks for the requested backtest")

test_weeks = all_weeks[-TEST_WEEKS:]
train_weeks = all_weeks[:-TEST_WEEKS]

train_sku = weekly_sku[weekly_sku["week_end"].isin(train_weeks)].copy()
test_sku = weekly_sku[weekly_sku["week_end"].isin(test_weeks)].copy()
train_subgroup = weekly_subgroup[
    weekly_subgroup["week_end"].isin(train_weeks)
].copy()
test_subgroup = weekly_subgroup[
    weekly_subgroup["week_end"].isin(test_weeks)
].copy()

performance_rows = []
subgroup_keys = train_subgroup[["Group", "Sub-Group"]].drop_duplicates()

for group_name, subgroup_name in subgroup_keys.itertuples(index=False, name=None):
    history = (
        train_subgroup.loc[
            (train_subgroup["Group"] == group_name)
            & (train_subgroup["Sub-Group"] == subgroup_name)
        ]
        .sort_values("week_end")["subgroup_sales_units"]
        .to_numpy()
    )
    actual = (
        test_subgroup.loc[
            (test_subgroup["Group"] == group_name)
            & (test_subgroup["Sub-Group"] == subgroup_name)
        ]
        .sort_values("week_end")["subgroup_sales_units"]
        .to_numpy()
    )

    for model in MODELS:
        prediction = forecast_weekly(history, len(actual), model)
        performance_rows.append({
            "Group": group_name,
            "Sub-Group": subgroup_name,
            "model": model,
            "actual_total": actual.sum(),
            "forecast_total": prediction.sum(),
            "absolute_error_total": np.abs(actual - prediction).sum(),
        })

subgroup_performance = pd.DataFrame(performance_rows)
model_summary = (
    subgroup_performance.groupby("model", as_index=False)[
        ["actual_total", "forecast_total", "absolute_error_total"]
    ].sum()
)
model_summary["wape"] = np.where(
    model_summary["actual_total"] > 0,
    model_summary["absolute_error_total"] / model_summary["actual_total"],
    np.nan,
)
model_summary["forecast_bias"] = np.where(
    model_summary["actual_total"] > 0,
    (model_summary["forecast_total"] - model_summary["actual_total"])
    / model_summary["actual_total"],
    np.nan,
)
model_summary["forecast_accuracy"] = (1 - model_summary["wape"]).clip(lower=0)
model_summary = model_summary.sort_values("wape").reset_index(drop=True)

best_model = model_summary.loc[0, "model"]
print("Train through:", max(train_weeks))
print("Test weeks:", pd.to_datetime(test_weeks).date)
print("Selected model:", best_model)
model_summary

## 4. Backtest the top-down SKU forecast

In [ ]:
# Shares are built strictly from the training period to avoid leakage.
backtest_shares = build_sku_shares(
    train_sku,
    recent_weeks=RECENT_SHARE_WEEKS,
    recent_weight=RECENT_SHARE_WEIGHT,
)

subgroup_forecast_rows = []
for group_name, subgroup_name in subgroup_keys.itertuples(index=False, name=None):
    history = (
        train_subgroup.loc[
            (train_subgroup["Group"] == group_name)
            & (train_subgroup["Sub-Group"] == subgroup_name)
        ]
        .sort_values("week_end")["subgroup_sales_units"]
        .to_numpy()
    )
    prediction = forecast_weekly(history, len(test_weeks), best_model)
    for week_end, value in zip(test_weeks, prediction):
        subgroup_forecast_rows.append({
            "week_end": pd.Timestamp(week_end),
            "Group": group_name,
            "Sub-Group": subgroup_name,
            "subgroup_forecast_units": max(0.0, float(value)),
        })

subgroup_backtest_forecast = pd.DataFrame(subgroup_forecast_rows)
sku_backtest_forecast = subgroup_backtest_forecast.merge(
    backtest_shares, on=["Group", "Sub-Group"], how="left"
)
sku_backtest_forecast["sku_forecast_units"] = (
    sku_backtest_forecast["subgroup_forecast_units"]
    * sku_backtest_forecast["allocation_share"]
)

sku_backtest = (
    test_sku.rename(columns={"sales_units": "actual_sales_units"})
    .merge(
        sku_backtest_forecast,
        on=["week_end", "Group", "Sub-Group", "product_id"],
        how="left",
        validate="one_to_one",
    )
)
if sku_backtest["sku_forecast_units"].isna().any():
    raise ValueError("Missing SKU forecasts after allocation")

sku_backtest["absolute_error"] = (
    sku_backtest["actual_sales_units"] - sku_backtest["sku_forecast_units"]
).abs()
actual_total = sku_backtest["actual_sales_units"].sum()
sku_wape = sku_backtest["absolute_error"].sum() / actual_total
sku_bias = (
    sku_backtest["sku_forecast_units"].sum() - actual_total
) / actual_total

backtest_summary = pd.DataFrame([{
    "selected_model": best_model,
    "subgroup_wape": model_summary.loc[0, "wape"],
    "subgroup_forecast_bias": model_summary.loc[0, "forecast_bias"],
    "sku_topdown_wape": sku_wape,
    "sku_topdown_forecast_bias": sku_bias,
    "sku_topdown_accuracy": max(0.0, 1 - sku_wape),
}])

print("Reconciliation maximum difference:", (
    sku_backtest_forecast.groupby(
        ["week_end", "Group", "Sub-Group"]
    )["sku_forecast_units"].sum()
    - subgroup_backtest_forecast.set_index(
        ["week_end", "Group", "Sub-Group"]
    )["subgroup_forecast_units"]
).abs().max())
backtest_summary

## 5. Forecast the next two weeks and save deliverables

In [ ]:
future_shares = build_sku_shares(
    weekly_sku,
    recent_weeks=RECENT_SHARE_WEEKS,
    recent_weight=RECENT_SHARE_WEIGHT,
)

last_week = weekly_sku["week_end"].max()
future_weeks = [
    last_week + pd.Timedelta(days=7 * step)
    for step in range(1, FORECAST_HORIZON_WEEKS + 1)
]

future_subgroup_rows = []
full_subgroup_keys = weekly_subgroup[["Group", "Sub-Group"]].drop_duplicates()
for group_name, subgroup_name in full_subgroup_keys.itertuples(index=False, name=None):
    history = (
        weekly_subgroup.loc[
            (weekly_subgroup["Group"] == group_name)
            & (weekly_subgroup["Sub-Group"] == subgroup_name)
        ]
        .sort_values("week_end")["subgroup_sales_units"]
        .to_numpy()
    )
    prediction = forecast_weekly(
        history, FORECAST_HORIZON_WEEKS, best_model
    )
    for week_end, value in zip(future_weeks, prediction):
        future_subgroup_rows.append({
            "week_end": week_end,
            "Group": group_name,
            "Sub-Group": subgroup_name,
            "subgroup_forecast_units": max(0.0, float(value)),
            "selected_model": best_model,
        })

future_subgroup = pd.DataFrame(future_subgroup_rows)
forecast_topdown = future_subgroup.merge(
    future_shares, on=["Group", "Sub-Group"], how="left"
)
forecast_topdown["sku_forecast_units"] = (
    forecast_topdown["subgroup_forecast_units"]
    * forecast_topdown["allocation_share"]
)

reconciliation = (
    forecast_topdown.groupby(
        ["week_end", "Group", "Sub-Group"], as_index=False
    ).agg(
        subgroup_forecast_units=("subgroup_forecast_units", "first"),
        allocated_sku_forecast_units=("sku_forecast_units", "sum"),
    )
)
reconciliation["difference"] = (
    reconciliation["subgroup_forecast_units"]
    - reconciliation["allocated_sku_forecast_units"]
)
max_reconciliation_difference = reconciliation["difference"].abs().max()
if max_reconciliation_difference > 1e-6:
    raise ValueError(
        f"Forecasts do not reconcile; maximum difference={max_reconciliation_difference}"
    )

forecast_topdown.to_csv(FORECAST_TOPDOWN_FILE, index=False)
model_summary.to_csv(OUTPUT_DIR / "hierarchical_model_performance.csv", index=False)
subgroup_performance.to_csv(
    OUTPUT_DIR / "hierarchical_subgroup_performance.csv", index=False
)
sku_backtest.to_csv(OUTPUT_DIR / "hierarchical_sku_backtest.csv", index=False)
backtest_summary.to_csv(OUTPUT_DIR / "hierarchical_backtest_summary.csv", index=False)

print("Future weeks:", pd.to_datetime(future_weeks).date)
print("Forecast rows:", len(forecast_topdown))
print("Products:", forecast_topdown["product_id"].nunique())
print("Maximum reconciliation difference:", max_reconciliation_difference)
print("Saved to:", OUTPUT_DIR)
forecast_topdown.head(10)